# 02 — Baseline XGBoost
> **LAB-VET-01 — estudo com PaySim (dados sintéticos).** Não use dados confidenciais, credenciais ou identificadores reais neste repositório. Execute as células em ordem. Os resultados históricos estão documentados separadamente; as saídas foram removidas para publicação.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Coloque o CSV público PaySim em data/ (não versionar arquivos de dados).
DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset ausente: {DATA_PATH.resolve()}. Consulte o README.")
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)

In [ ]:
treino = df.loc[df["step"] <= 300].copy()
validacao = df.loc[(df["step"] > 300) & (df["step"] <= 400)].copy()
teste = df.loc[df["step"] > 400].copy()
print("Treino:", treino.shape, "Validação:", validacao.shape, "Teste:", teste.shape)

In [ ]:
features_baseline = ["amount", "oldbalanceOrg", "oldbalanceDest", "step", "type_CASH_IN", "type_CASH_OUT", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"]
def montar_X(dados, colunas):
    return pd.get_dummies(dados[["amount", "oldbalanceOrg", "oldbalanceDest", "step", "type"]], columns=["type"], dtype="int8").reindex(columns=colunas, fill_value=0)

In [ ]:
from xgboost import XGBClassifier
def novo_modelo():
    return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", tree_method="hist", random_state=42, n_jobs=-1)

## Treinamento — apenas dados de treino

In [ ]:
X_train = montar_X(treino, features_baseline)
X_valid = montar_X(validacao, features_baseline)
y_train = treino["isFraud"]
y_valid = validacao["isFraud"]
modelo_baseline = novo_modelo()
modelo_baseline.fit(X_train, y_train)
prob_valid = modelo_baseline.predict_proba(X_valid)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

def avaliar(y, prob):
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0), "average_precision": average_precision_score(y, prob), "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}

def avaliar_top_k(y, prob, k):
    labels = np.asarray(y)
    idx = np.argsort(-np.asarray(prob), kind="stable")[:k]
    tp = int(labels[idx].sum())
    return {"Top_K": k, "TP": tp, "FP": len(idx)-tp, "precision": tp/len(idx), "recall": tp/int(labels.sum())}

In [ ]:
print(pd.Series(avaliar(y_valid, prob_valid)))
print(pd.DataFrame([avaliar_top_k(y_valid, prob_valid, k) for k in [250,500,750,1000,1250]]))

**Nota:** não utilizar saldos posteriores, `isFraud` ou `isFlaggedFraud` como preditores.

---
**Autoria**

```text
Alexandre Lotti
<!LOTTI>
```
